In [2]:
# ================================================================
# SINGLE JUPYTER CELL
# Improved DisWOT on ImageNet-16-120
#
# Your improvements:
#   1. Hybrid feature loss = L2 + InfoNCE
#   2. KL divergence for logits
#   3. Grad-CAM++-style attention alignment
#   4. Learnable temperature with Softplus
#   5. Metrics: Loss, Top-1, Top-5, Accuracy, Precision, Recall, F1
# ================================================================

# -----------------------------
# 0. INSTALL DEPENDENCIES
# -----------------------------
import sys, subprocess, os, random, time, math, urllib.request
from pathlib import Path

def pip_install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

try:
    import torch
    import torchvision
except:
    pip_install("torch")
    pip_install("torchvision")

try:
    import sklearn
except:
    pip_install("scikit-learn")

try:
    import gdown
except:
    pip_install("gdown")

# -----------------------------
# 1. IMPORTS
# -----------------------------
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T

from torch.utils.data import DataLoader, Subset
from torchvision.models import resnet18, resnet50

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import numpy as np

# -----------------------------
# 2. CONFIGURATION
# -----------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print("Improved DisWOT - ImageNet-16-120")
print("=" * 70)
print("Device:", DEVICE)

# Dataset
DATA_ROOT = "./data"

# ------------------------------------------------
# For a quick experiment:
# Set to None for the complete dataset.
# ------------------------------------------------
MAX_TRAIN = 10000
MAX_TEST  = 3000

# Training
EPOCHS = 10
BATCH_SIZE = 128
LR = 0.01
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4

# Improved DisWOT loss weights
LAMBDA_CE = 1.0
LAMBDA_FEATURE = 1.0
LAMBDA_INFONCE = 0.5
LAMBDA_KL = 1.0
LAMBDA_ATTENTION = 0.5

# Initial KD temperature
INITIAL_TEMPERATURE = 4.0

# InfoNCE temperature
INFONCE_TEMPERATURE = 0.07

NUM_CLASSES = 120

# -----------------------------
# 3. DOWNLOAD / INSTALL
# -----------------------------
#
# The recommended ImageNet16-120 loader is provided
# by AutoDL-Projects/NAS-Bench-201.
#
# We install the loader directly from GitHub.
# -----------------------------

print("\n[1] Preparing ImageNet-16-120 loader...")

repo_dir = Path("./AutoDL-Projects")

if not repo_dir.exists():
    print("Downloading AutoDL-Projects...")
    subprocess.run(
        [
            "git", "clone",
            # Removed --depth 1 for more robust clone of submodules
            "https://github.com/D-X-Y/AutoDL-Projects.git",
            str(repo_dir)
        ],
        check=True
    )

    # Initialize and update submodules to ensure all dependencies are fetched
    print("Initializing Git submodules...")
    original_cwd = os.getcwd()
    os.chdir(repo_dir)
    try:
        subprocess.check_call(
            ["git", "submodule", "update", "--init", "--recursive"]
        )
    finally:
        os.chdir(original_cwd)

# Add repository to Python path
sys.path.insert(0, str(repo_dir))

# Verify if the expected ImageNet16.py file exists after cloning
expected_imagenet_path = repo_dir / "xautodl" / "datasets" / "ImageNet16.py"
if not expected_imagenet_path.exists():
    print(f"Warning: ImageNet16.py not found at expected path: {expected_imagenet_path}")
    print("This might indicate an issue with the git clone or repository structure.")

# Try importing the official ImageNet16 loader
ImageNet16 = None

possible_modules = [
    "lib.datasets.ImageNet16",
    "xautodl.datasets.ImageNet16",
]

for module_name in possible_modules:
    try:
        module = __import__(
            module_name,
            fromlist=["ImageNet16"]
        )
        ImageNet16 = module.ImageNet16
        break
    except Exception:
        pass

if ImageNet16 is None:

    # Search repository automatically as a fallback
    candidate = None

    for root, dirs, files in os.walk(repo_dir):
        for file in files:
            if file.lower() in [
                "imagenet16.py",
                "imagenet16_120.py"
            ]:
                candidate = os.path.join(root, file)
                break
        if candidate:
            break

    if candidate is not None:
        import importlib.util

        spec = importlib.util.spec_from_file_location(
            "imagenet16_module",
            candidate
        )

        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)

        if hasattr(mod, 'ImageNet16'):
            ImageNet16 = mod.ImageNet16
        else:
            raise RuntimeError(f"Class 'ImageNet16' not found in {candidate}")

if ImageNet16 is None:
    raise RuntimeError(
        "Could not find ImageNet16 loader. "
        "Please check the AutoDL-Projects repository."
    )

print("ImageNet16 loader: OK")

# -----------------------------
# 4. DATASET LOCATION
# -----------------------------
#
# Official projects use an ImageNet16 directory
# under the dataset root.
# -----------------------------

os.makedirs(DATA_ROOT, exist_ok=True)

IMAGE_NET16_ROOT = os.path.join(
    DATA_ROOT,
    "ImageNet16"
)

os.makedirs(IMAGE_NET16_ROOT, exist_ok=True)

print("\nDataset root:")
print(os.path.abspath(IMAGE_NET16_ROOT))

# ------------------------------------------------
# IMPORTANT:
#
# ImageNet-16-120 is distributed through the
# NAS-Bench-201/NATS-Bench Google Drive dataset.
#
# If it is not already present, the cell attempts
# to download it through gdown.
#
# ------------------------------------------------

def find_imagenet16(root):
    """
    Search recursively for files/directories indicating
    an ImageNet16 dataset.
    """
    for r, d, f in os.walk(root):
        for name in f:
            low = name.lower()
            if (
                "imagenet16" in low
                or "imagenet-16" in low
            ):
                return os.path.join(r, name)

        for name in d:
            low = name.lower()
            if "imagenet16" in low:
                return os.path.join(r, name)

    return None

existing_dataset = find_imagenet16(DATA_ROOT)

if existing_dataset:
    print("Existing ImageNet16 dataset found:")
    print(existing_dataset)

else:
    print("\nImageNet16 dataset was not found locally.")
    print(
        "The official NAS-Bench-201/NATS-Bench distribution "
        "is normally obtained from Google Drive."
    )

    # ---------------------------------------------------------
    # Google Drive folder used by several NAS-Bench repositories
    # ---------------------------------------------------------
    DRIVE_FOLDER = (
        "https://drive.google.com/drive/folders/"
        "1L0Lzq8rWpZLPfiQGd6QR8q5xLV88emU7"
    )

    print("\nDownloading dataset from:")
    print(DRIVE_FOLDER)

    try:
        import gdown

        gdown.download_folder(
            DRIVE_FOLDER,
            output=DATA_ROOT,
            quiet=False,
            use_cookies=False
        )

    except Exception as e:

        print("\nAutomatic Google Drive download failed.")
        print("Error:", e)

        print(
            "\nPlease download ImageNet16 manually from the "
            "NAS-Bench-201 dataset source and place it here:"
        )

        print(
            os.path.abspath(IMAGE_NET16_ROOT)
        )

        raise RuntimeError(
            "\nDataset download failed. "
            "After placing ImageNet16 under ./data/ImageNet16, "
            "run this cell again."
        )

# -----------------------------
# 5. TRANSFORMS
# -----------------------------
#
# ImageNet16 images are already 16x16.
# We resize to 64x64 because standard ResNet
# architectures are designed for larger feature maps.
# -----------------------------

train_transform = T.Compose([
    T.Resize((64, 64)),
    T.RandomHorizontalFlip(),
    T.RandomCrop(64, padding=4),
    T.ToTensor(),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = T.Compose([
    T.Resize((64, 64)),
    T.ToTensor(),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# -----------------------------
# 6. LOAD DATASET
# -----------------------------
print("\n[2] Loading ImageNet-16-120...")

try:

    train_dataset = ImageNet16(
        IMAGE_NET16_ROOT,
        True,
        train_transform,
        120
    )

    test_dataset = ImageNet16(
        IMAGE_NET16_ROOT,
        False,
        test_transform,
        120
    )

except Exception as e:

    print("\nDataset loading failed.")
    print("Error:", e)

    print(
        "\nExpected structure is the ImageNet16 data "
        "used by NAS-Bench-201."
    )

    raise

print("Full training samples:", len(train_dataset))
print("Full test samples:", len(test_dataset))
print("Classes:", NUM_CLASSES)

# -----------------------------
# 7. REDUCE DATASET FOR TEST
# -----------------------------
if MAX_TRAIN is not None:
    n = min(MAX_TRAIN, len(train_dataset))
    train_dataset = Subset(
        train_dataset,
        range(n)
    )

if MAX_TEST is not None:
    n = min(MAX_TEST, len(test_dataset))
    test_dataset = Subset(
        test_dataset,
        range(n)
    )

print("Experiment training samples:", len(train_dataset))
print("Experiment test samples:", len(test_dataset))

# -----------------------------
# 8. DATALOADERS
# -----------------------------
num_workers = min(4, os.cpu_count() or 1)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=(DEVICE.type == "cuda")
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=(DEVICE.type == "cuda")
)

# -----------------------------
# 9. TEACHER
# -----------------------------
print("\n[3] Creating Teacher / Student...")

teacher = resnet50(weights=None)
teacher.fc = nn.Linear(
    teacher.fc.in_features,
    NUM_CLASSES
)

# -----------------------------
# 10. STUDENT
# -----------------------------
student = resnet18(weights=None)
student.fc = nn.Linear(
    student.fc.in_features,
    NUM_CLASSES
)

teacher = teacher.to(DEVICE)
student = student.to(DEVICE)

# Teacher is frozen for KD
for p in teacher.parameters():
    p.requires_grad = False

teacher.eval()

print(
    "Teacher parameters:",
    sum(p.numel() for p in teacher.parameters())
)

print(
    "Student parameters:",
    sum(p.numel() for p in student.parameters())
)

# -----------------------------
# 11. FEATURE ADAPTER
# -----------------------------
#
# ResNet50 final feature = 2048
# ResNet18 final feature = 512
#
# Project student feature to teacher dimension.
# -----------------------------

feature_adapter = nn.Linear(
    512,
    2048
).to(DEVICE)

# -----------------------------
# 12. LEARNABLE TEMPERATURE
# -----------------------------
#
# T = Softplus(raw_temperature)
# guarantees T > 0.
# -----------------------------

raw_temperature = nn.Parameter(
    torch.tensor(
        math.log(
            math.exp(INITIAL_TEMPERATURE) - 1
        ),
        device=DEVICE
    )
)

# -----------------------------
# 13. FEATURE EXTRACTION
# -----------------------------

teacher_features = None
student_features = None

def get_resnet_feature(model, x):

    x = model.conv1(x)
    x = model.bn1(x)
    x = model.relu(x)
    x = model.maxpool(x)

    x = model.layer1(x)
    x = model.layer2(x)
    x = model.layer3(x)
    feature_map = model.layer4(x)

    pooled = F.adaptive_avg_pool2d(
        feature_map,
        1
    )

    pooled = torch.flatten(
        pooled,
        1
    )

    logits = model.fc(pooled)

    return logits, pooled, feature_map

# -----------------------------
# 14. L2 FEATURE LOSS
# -----------------------------

def l2_feature_loss(student_f, teacher_f):

    student_f = F.normalize(
        student_f,
        dim=1
    )

    teacher_f = F.normalize(
        teacher_f,
        dim=1
    )

    return F.mse_loss(
        student_f,
        teacher_f
    )

# -----------------------------
# 15. INFO NCE LOSS
# -----------------------------

def info_nce_loss(
    student_f,
    teacher_f,
    temperature=0.07
):

    student_f = F.normalize(
        student_f,
        dim=1
    )

    teacher_f = F.normalize(
        teacher_f,
        dim=1
    )

    similarity = torch.matmul(
        student_f,
        teacher_f.T
    )

    similarity = similarity / temperature

    labels = torch.arange(
        student_f.size(0),
        device=student_f.device
    )

    return F.cross_entropy(
        similarity,
        labels
    )

# -----------------------------
# 16. KL LOGIT LOSS
# -----------------------------

def kd_kl_loss(
    student_logits,
    teacher_logits,
    temperature
):

    student_log_prob = F.log_softmax(
        student_logits / temperature,
        dim=1
    )

    teacher_prob = F.softmax(
        teacher_logits / temperature,
        dim=1
    )

    return F.kl_div(
        student_log_prob,
        teacher_prob,
        reduction="batchmean"
    ) * (temperature ** 2)

# -----------------------------
# 17. GRAD-CAM++-STYLE ATTENTION
# -----------------------------
#
# We calculate gradient-based attention over
# the final convolutional feature map.
#
# This is a lightweight implementation suitable
# for the single-cell experiment.
# -----------------------------

def gradcam_pp_attention(
    feature_map,
    logits,
    labels
):

    selected = logits[
        torch.arange(
            logits.size(0),
            device=logits.device
        ),
        labels
    ].sum()

    gradients = torch.autograd.grad(
        selected,
        feature_map,
        retain_graph=True,
        create_graph=False,
        allow_unused=True
    )[0]

    if gradients is None:
        return torch.zeros(
            feature_map.size(
                0
            ),
            1,
            feature_map.size(2),
            feature_map.size(3),
            device=feature_map.device
        )

    gradients = gradients.detach()

    # Positive gradients
    positive_gradients = F.relu(
        gradients
    )

    # Grad-CAM++ approximation:
    # second-order weighting
    grad_sq = positive_gradients ** 2

    grad_cube = positive_gradients ** 3

    denominator = (
        2.0 * grad_sq
        + grad_cube.mean(
            dim=(2, 3),
            keepdim=True
        )
    )

    denominator = denominator + 1e-8

    alpha = grad_sq / denominator

    weights = (
        alpha *
        positive_gradients
    ).sum(
        dim=(2, 3),
        keepdim=True
    )

    cam = (
        weights *
        feature_map.detach()
    ).sum(
        dim=1,
        keepdim=True
    )

    cam = F.relu(cam)

    cam = F.interpolate(
        cam,
        size=(8, 8),
        mode="bilinear",
        align_corners=False
    )

    cam = cam.flatten(1)

    cam = F.normalize(
        cam,
        dim=1
    )

    return cam

# -----------------------------
# 18. ATTENTION LOSS
# -----------------------------

def attention_loss(
    student_attention,
    teacher_attention
):

    return F.mse_loss(
        student_attention,
        teacher_attention
    )

# -----------------------------
# 19. OPTIMIZER
# -----------------------------

optimizer = torch.optim.SGD(
    list(student.parameters())
    + list(feature_adapter.parameters())
    + [raw_temperature],
    lr=LR,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

# -----------------------------
# 20. TRAINING
# -----------------------------

print("\n[4] Starting Improved DisWOT training...")
print("=" * 70)

start_time = time.time()

for epoch in range(EPOCHS):

    student.train()

    running_loss = 0.0
    running_ce = 0.0
    running_feature = 0.0
    running_infonce = 0.0
    running_kl = 0.0
    running_attention = 0.0

    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        # ------------------------------------------------
        # Teacher
        # ------------------------------------------------
        with torch.no_grad():

            t_logits, t_features, t_map = \
                get_resnet_feature(
                    teacher,
                    images
                )

        # ------------------------------------------------
        # Student
        # ------------------------------------------------

        s_logits, s_features, s_map = \
            get_resnet_feature(
                student,
                images
            )

        # ------------------------------------------------
        # Feature projection
        # ------------------------------------------------

        s_projected = feature_adapter(
            s_features
        )

        # ------------------------------------------------
        # 1. Classification loss
        # ------------------------------------------------

        loss_ce = F.cross_entropy(
            s_logits,
            labels
        )

        # ------------------------------------------------
        # 2. L2 feature alignment
        # ------------------------------------------------

        loss_l2 = l2_feature_loss(
            s_projected,
            t_features
        )

        # ------------------------------------------------
        # 3. InfoNCE
        # ------------------------------------------------

        loss_infonce = info_nce_loss(
            s_projected,
            t_features,
            INFONCE_TEMPERATURE
        )

        # ------------------------------------------------
        # 4. Learnable temperature
        # ------------------------------------------------

        temperature = F.softplus(
            raw_temperature
        ) + 1e-4

        # ------------------------------------------------
        # 5. KL logits
        # ------------------------------------------------

        loss_kl = kd_kl_loss(
            s_logits,
            t_logits,
            temperature
        )

        # ------------------------------------------------
        # 6. Attention alignment
        #
        # We use predicted labels for both networks.
        # ------------------------------------------------

        student_attention = gradcam_pp_attention(
            s_map,
            s_logits,
            labels
        )

        # Teacher attention is calculated without
        # gradient tracking around the teacher weights,
        # but gradient wrt feature map is needed.
        t_map_for_cam = t_map.detach().requires_grad_(True)

        t_selected = t_logits[
            torch.arange(
                t_logits.size(0),
                device=DEVICE
            ),
            labels
        ].sum()

        t_grad = torch.autograd.grad(
            t_selected,
            t_map,
            retain_graph=True,
            allow_unused=True
        )[0]

        if t_grad is None:
            teacher_attention = torch.zeros_like(
                student_attention
            )
        else:

            t_positive = F.relu(
                t_grad.detach()
            )

            t_cam = (
                t_positive *
                t_map.detach()
            ).sum(
                dim=1,
                keepdim=True
            )

            t_cam = F.relu(
                t_cam
            )

            t_cam = F.interpolate(
                t_cam,
                size=(8, 8),
                mode="bilinear",
                align_corners=False
            )

            t_cam = t_cam.flatten(1)

            teacher_attention = F.normalize(
                t_cam,
                dim=1
            )

        loss_attention = attention_loss(
            student_attention,
            teacher_attention
        )

        # ------------------------------------------------
        # HYBRID TOTAL LOSS
        # ------------------------------------------------

        loss_feature = (
            LAMBDA_FEATURE * loss_l2
            + LAMBDA_INFONCE * loss_infonce
        )

        total_loss = (
            LAMBDA_CE * loss_ce
            + loss_feature
            + LAMBDA_KL * loss_kl
            + LAMBDA_ATTENTION * loss_attention
        )

        # ------------------------------------------------
        # Backpropagation
        # ------------------------------------------------

        total_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            list(student.parameters())
            + list(feature_adapter.parameters())
            + [raw_temperature],
            max_norm=5.0
        )

        optimizer.step()

        # ------------------------------------------------
        # Statistics
        # ------------------------------------------------

        running_loss += total_loss.item()
        running_ce += loss_ce.item()
        running_feature += loss_l2.item()
        running_infonce += loss_infonce.item()
        running_kl += loss_kl.item()
        running_attention += loss_attention.item()

        _, predicted = torch.max(
            s_logits,
            1
        )

        total += labels.size(0)

        correct += (
            predicted == labels
        ).sum().item()

    scheduler.step()

    train_acc = (
        100.0 *
        correct /
        max(total, 1)
    )

    avg_loss = (
        running_loss /
        len(train_loader)
    )

    print(
        f"Epoch [{epoch+1:03d}/{EPOCHS}] "
        f"Loss={avg_loss:.4f} | "
        f"CE={running_ce/len(train_loader):.4f} | "
        f"L2={running_feature/len(train_loader):.4f} | "
        f"InfoNCE={running_infonce/len(train_loader):.4f} | "
        f"KL={running_kl/len(train_loader):.4f} | "
        f"Att={running_attention/len(train_loader):.4f} | "
        f"Temp={temperature.item():.3f} | "
        f"Train Acc={train_acc:.2f}%"
    )

# -----------------------------
# 21. EVALUATION
# -----------------------------

print("\n[5] Final evaluation")
print("=" * 70)

student.eval()

all_labels = []
all_predictions = []
all_probabilities = []

test_loss = 0.0

correct_top1 = 0
correct_top5 = 0
total_test = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        logits, _, _ = get_resnet_feature(
            student,
            images
        )

        loss = F.cross_entropy(
            logits,
            labels
        )

        test_loss += loss.item()

        probabilities = F.softmax(
            logits,
            dim=1
        )

        # Top-1
        _, pred = torch.max(
            logits,
            1
        )

        correct_top1 += (
            pred == labels
        ).sum().item()

        # Top-5
        top5 = torch.topk(
            logits,
            k=5,
            dim=1
        ).indices

        correct_top5 += (
            top5 == labels.unsqueeze(1)
        ).any(
            dim=1
        ).sum().item()

        total_test += labels.size(0)

        all_labels.extend(
            labels.cpu().numpy()
        )

        all_predictions.extend(
            pred.cpu().numpy()
        )

        all_probabilities.append(
            probabilities.cpu()
        )

# -----------------------------
# 22. METRICS
# -----------------------------

avg_test_loss = (
    test_loss /
    len(test_loader)
)

top1 = (
    100.0 *
    correct_top1 /
    total_test
)

top5 = (
    100.0 *
    correct_top5 /
    total_test
)

accuracy = accuracy_score(
    all_labels,
    all_predictions
) * 100

precision = precision_score(
    all_labels,
    all_predictions,
    average="macro",
    zero_division=0
) * 100

recall = recall_score(
    all_labels,
    all_predictions,
    average="macro",
    zero_division=0
) * 100

f1 = f1_score(
    all_labels,
    all_predictions,
    average="macro",
    zero_division=0
) * 100

elapsed = (
    time.time() -
    start_time
)

# -----------------------------
# 23. FINAL RESULTS
# -----------------------------

print("\n")
print("=" * 70)
print("        IMPROVED DISWOT - FINAL RESULTS")
print("=" * 70)

print(f"Dataset              : ImageNet-16-120")
print(f"Number of classes    : {NUM_CLASSES}")
print(f"Train samples        : {len(train_dataset)}")
print(f"Test samples         : {len(test_dataset)}")

print("-" * 70)

print(
    f"Test Loss             : {avg_test_loss:.4f}"
)

print(
    f"Top-1 Accuracy        : {top1:.2f}%"
)

print(
    f"Top-5 Accuracy        : {top5:.2f}%"
)

print(
    f"Accuracy              : {accuracy:.2f}%"
)

print(
    f"Macro Precision       : {precision:.2f}%"
)

print(
    f"Macro Recall          : {recall:.2f}%"
)

print(
    f"Macro F1              : {f1:.2f}%"
)

print("-" * 70)

print(
    f"Final Temperature     : "
    f"{F.softplus(raw_temperature).item():.4f}"
)

print(
    f"Training Time         : "
    f"{elapsed/60:.2f} minutes"
)

print("=" * 70)

# -----------------------------
# 24. SAVE STUDENT
# -----------------------------

torch.save(
    {
        "student": student.state_dict(),
        "feature_adapter": feature_adapter.state_dict(),
        "temperature": raw_temperature.detach().cpu(),
        "metrics": {
            "loss": avg_test_loss,
            "top1": top1,
            "top5": top5,
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1
        }
    },
    "Improved_DisWOT_ImageNet16_120.pth"
)

print(
    "\nModel saved as: "
    "Improved_DisWOT_ImageNet16_120.pth"
)

Improved DisWOT - ImageNet-16-120
Device: cpu

[1] Preparing ImageNet-16-120 loader...


RuntimeError: Could not find ImageNet16 loader. Please check the AutoDL-Projects repository.